# 14 · Contexto jurídico I: a língua do Direito e o deslocamento de domínio

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


**Pergunta deste notebook:** o que acontece com tudo o que vimos (coocorrência, word2vec, fastText, avaliação, analogias) quando o corpus é **texto jurídico brasileiro**? Comparamos sistematicamente um embedding **geral** (Wikipedia2Vec PT), um **literário** (Machado de Assis) e embeddings **jurídicos** treinados aqui em decisões do STF.

Leitura de apoio: [`docs/juridico.md`](../docs/juridico.md) (peculiaridades da língua do Direito, particularidades brasileiras, aplicações e debates).

## Corpora jurídicos públicos usados
| Corpus | Conteúdo | Uso aqui |
|---|---|---|
| **RulingBR** (Feijó & Moreira, 2018) | 10.574 decisões do STF (2011–2018): ementa, relatório, voto, área, classe, relator | treino dos embeddings jurídicos (6 M tokens); classificação de área |
| **LeNER-Br** (Luz de Araujo et al., 2018) | decisões anotadas com PESSOA, ORGANIZACAO, LOCAL, TEMPO, LEGISLACAO, JURISPRUDENCIA | avaliação extrínseca: NER jurídico |
| **JurisBERT STS** (Viegas et al., 2023) | pares de ementas (STJ, TJMS) rotulados como similares ou não | avaliação extrínseca: similaridade de ementas |

Os modelos `legal_sg100`, `legal_cbow100` e `legal_ft100` são treinados por `uv run replang train legal` com **o mesmo pré-processamento e hiperparâmetros** dos modelos Machado — para que a comparação isole o *domínio*.

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab renderiza offline; HTML carrega plotly.js via CDN (notebooks leves)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: Y:\BLIS\linguagemBLIS2S26\linguagem-blis2s26 | fast: False


## 1. Perfil dos corpora: a língua do Direito em números

Comparamos o corpus jurídico (amostra de 1,5 M tokens, mesma ordem de grandeza do Machado) com o literário: tamanho de sentença, diversidade lexical, numerais, latim, a palavra *art*.

In [2]:
from replang.data.corpora import load_machado_sentences
from replang.data.legal import load_legal_sentences, corpus_profile, LEGAL
mach = load_machado_sentences()
jur = load_legal_sentences(prefer_full=False)   # amostra versionada (1,5 M tokens)
prof = pd.DataFrame([corpus_profile(mach, "Machado de Assis (literário)"), corpus_profile(jur, "STF / RulingBR (jurídico)")]).set_index("corpus").T
prof.style.format("{:,.4f}")

corpus,Machado de Assis (literário),STF / RulingBR (jurídico)
sentenças,"150,091.0000","64,659.0000"
tokens,"2,357,192.0000","1,500,009.0000"
tipos,"74,820.0000","31,974.0000"
TTR (tipos/tokens),0.0317,0.0213
tokens/sentença (média),15.7051,23.1988
sentenças > 40 tokens,0.0330,0.1512
numerais (token '0…'),0.0015,0.0349
'art',0.0000,0.0048
latim (tokens),0.0016,0.0045
hapax (freq. 1),0.4158,0.3613


In [3]:
import plotly.express as px
lens = pd.DataFrame({"tokens por sentença": [len(s) for s in mach[:40000]] + [len(s) for s in jur[:40000]], "corpus": ["literário"] * 40000 + ["jurídico"] * 40000})
px.histogram(lens[lens["tokens por sentença"] <= 120], x="tokens por sentença", color="corpus", barmode="overlay", nbins=60, histnorm="probability", template="plotly_white",
             title="Distribuição do tamanho das sentenças (≤ 120 tokens)").show()

Dois traços saltam: as sentenças jurídicas são **mais longas** (cauda pesada acima de 40 tokens — períodos subordinados, enumerações) e os **numerais** são muito mais frequentes (artigos, leis, processos, datas, valores). Isso já afeta decisões de modelagem: janela de contexto, e o que fazer com números.

### 1.1 O pré-processamento de Hartmann et al. aplicado a texto jurídico

A normalização "numerais → 0" (notebook 00) é razoável em notícias; em Direito, **o número é o nome da norma**. Veja o que o pipeline faz com uma citação típica:

In [4]:
from replang.utils.text import normalize_text, tokenize
ex = "Nos termos do art. 5º, inciso LV, da CF/88 e do art. 1.022 do CPC (Lei nº 13.105/2015), conheço do REsp 1.234.567/SP e nego-lhe provimento, com fundamento na Súmula 7/STJ."
print("normalizado:", normalize_text(ex))
print("tokens     :", tokenize(ex))

normalizado: nos termos do art. 0º, inciso lv, da cf/00 e do art. 0.000 do cpc (lei nº 00.000/0000), conheço do resp 0.000.000/sp e nego-lhe provimento, com fundamento na súmula 0/stj.
tokens     : ['nos', 'termos', 'do', 'art', '0', 'º', 'inciso', 'lv', 'da', 'cf', '00', 'e', 'do', 'art', '0.000', 'do', 'cpc', 'lei', 'nº', '00.000/0000', 'conheço', 'do', 'resp', '0.000.000/', 'sp', 'e', 'nego-lhe', 'provimento', 'com', 'fundamento', 'na', 'súmula', '0/', 'stj']


*Lei 13.105/2015* vira *lei 0.000/0000*; *art. 5º* e *art. 1.022* viram o mesmo token; *§* desaparece (é pontuação); *CF/88* vira *cf/00*. Para **treinar embeddings de palavras** isso é aceitável (números são ruído distribucional); para **extrair entidades** (LeNER-Br: LEGISLACAO, JURISPRUDENCIA) é destrutivo. Uma boa prática de domínio: tokenizar números e siglas como tokens próprios *antes* de qualquer normalização, ou tratá-los como entidades.

## 2. Coocorrência no domínio: a radiografia do vocabulário técnico

A hipótese distribucional (notebook 01) aplicada ao corpus do STF: com que palavras *sentença*, *ação* e *parte* coocorrem? Compare com o que o leitor leigo esperaria.

In [5]:
from replang.models.cooccurrence import CountModel
cm = CountModel(window=4, dim=100, min_count=10, alpha=0.75, weighting="harmonic", max_vocab=20000).fit(jur)
ppmi = cm.ppmi_
for w in ["sentença", "ação", "parte", "título", "pena", "agravo"]:
    i = cm.index[w]
    row = ppmi[i].toarray().ravel()
    top = [cm.words[j] for j in np.argsort(-row)[:10]]
    print(f"{w:<9} PPMI mais alto com: {top}")

sentença  PPMI mais alto com: ['condenatória', 'exequenda', 'absolutória', 'transitada', 'prolação', 'recorrível', 'manteve', 'pronúncia', 'apelada', 'reformada']
ação      PPMI mais alto com: ['rescisória', 'direta', 'coletiva', 'propor', 'trancamento', 'declaratória', 'mandamental', 'ajuizou', 'revisional', 'nobre']
parte     PPMI mais alto com: ['dispositiva', 'insurgente', 'interessada', 'recorrente', 'autora', 'arcando', 'impertinentes', 'agravante', 'sugestiona', 'sucumbente']
título    PPMI mais alto com: ['precário', 'inexigibilidade', 'arbitrado', 'executivo', 'condenatório', 'seções', 'inexigível', 'adoto', 'judicial', 'imutabilidade']
pena      PPMI mais alto com: ['privativa', 'sob', 'substituição', 'imposta', 'cominada', 'individualização', 'dosimetria', 'diminuição', 'restritiva', 'corporal']
agravo    PPMI mais alto com: ['regimental', 'maneja', 'manejado', 'vistos', 'improvido', 'instrumento', 'tempestivo', 'desprovido', 'manejam', 'observaram-se']


## 3. Deslocamento de domínio: vizinhos no geral × no jurídico

Carregamos os embeddings e comparamos os vizinhos das palavras **polissêmicas** do léxico jurídico (`replang.data.legal.LEGAL.polysemous`). Como espaços diferentes não são comparáveis por cosseno, medimos a **sobreposição das vizinhanças** (Jaccard dos 10 vizinhos).

In [6]:
from replang.data.embeddings import load_ptwiki, load_local_model
from replang.data.legal import neighborhood_overlap
pt = load_ptwiki(); pt.name = "Wikipédia"
leg = load_local_model("legal_sg100"); leg.name = "STF"
mac = load_local_model("machado_sg100"); mac.name = "Machado"
print(pt, "|", leg, "|", mac)
for w in ["sentença", "ação", "parte", "título", "pena", "agravo", "competência", "prescrição", "trânsito", "remédio"]:
    print(f"\n{w}")
    for wv in (pt, leg, mac):
        print(f"   {wv.name:<9} {[x for x, _ in wv.most_similar(w, topn=7)] if w in wv else 'OOV'}")

WordVectors(name='Wikipédia', n=50000, dim=100) | WordVectors(name='STF', n=22203, dim=100) | WordVectors(name='Machado', n=23194, dim=100)

sentença
   Wikipédia ['sentenças', 'condenação', 'instância', 'nulidade', 'absolvição', 'julgamento', 'prisão']
   STF       ['condenatória', 'absolutória', 'transitada', 'homologatória', 'sobrevindo', 'reformou', 'confirmatório']
   Machado   ['interpretação', 'afirmação', 'falsa', 'diplomática', 'vencida', 'complicada', 'póstuma']

ação
   Wikipédia ['acção', 'intervenção', 'afirmativa', 'reação', 'atividade', 'liberação', 'aventura']
   STF       ['rescisória', 'intentada', 'ajuizada', 'possessória', 'movida', 'propositura', 'expropriatória']
   Machado   ['extensão', 'utilidade', 'esfera', 'perfeição', 'mental', 'invenção', 'individualidade']

parte
   Wikipédia ['restante', 'porção', 'exceção', 'parcela', 'resto', 'metade', 'fazia']
   STF       ['recorrente', 'adversa', 'insurgente', 'dissentido', 'infirmou', 'especificadamente', 'desincumb

In [7]:
probe = [w for w in LEGAL.probe_words if w in pt and w in leg]
shift = neighborhood_overlap(pt, leg, probe, k=10)
print(f"sobreposição média (Jaccard@10) entre Wikipédia e STF nas {len(probe)} palavras polissêmicas: {shift['jaccard@k'].mean():.3f}")
shift.head(15).style.format({"jaccard@k": "{:.2f}"})

sobreposição média (Jaccard@10) entre Wikipédia e STF nas 27 palavras polissêmicas: 0.008


,palavra,jaccard@k,vizinhos Wikipédia,vizinhos STF
0,sentença,0.00,"absolvição, acusação, condenação, condenações, condicional, instância","absolutória, condenatória, confirmatório, exequenda, homologatória, recorrível"
1,ação,0.00,"acção, afirmativa, atividade, aventura, contestação, intervenção","ajuizada, ajuizamento, declaratória, expropriatória, intentada, movida"
2,parte,0.00,"exceção, fazia, maior, metade, parcela, porção","adversa, agravante, desincumbido, desincumbiu, dissentido, especificadamente"
4,pena,0.00,"condenados, leytor, multa, penas, perpétua, prisão","estabeleço, inicie, privativa, readequação, reclusiva, reduzo"
7,prescrição,0.00,"abstinência, medicamento, medicamentos, medicação, obrigatoriedade, prescrito","executória, falimentar, intercorrente, operou-se, prescricional, punibilidade"
6,competência,0.00,"atribuições, competente, competências, disciplinar, efetividade, julgar","competir, declinação, invade, originária, processar, usurpa"
9,tutela,0.00,"alçada, auspícios, chefia, custódia, encargo, jurisdição","antecipada, antecipatória, antecipação, contracautela, denegam, inaudita"
8,mérito,0.00,"agraciado, benemerência, comenda, comendador, condecoração, diploma","adentrar, apreciá-lo, confirmando-se, determinar-se, extingo, implicada"
14,recurso,0.00,"aplicação, arquivamento, caso, dispositivo, modo, método","agravo, destrancamento, destrancar, extraordinário, inadmitir, interposto"
13,juízo,0.00,"delito, julga, julgamento, julgamentos, juízos, réu","declinou, emendatio, itaperuna, magistrada, negativo, primevo"


In [8]:
# Controle: palavras NÃO técnicas (frequentes nos dois corpora) devem ter mais sobreposição
comuns = [w for w in ["casa", "cidade", "tempo", "dia", "homem", "mulher", "trabalho", "família", "governo", "dinheiro", "saúde", "escola", "água", "carro", "livro"] if w in pt and w in leg]
ctrl = neighborhood_overlap(pt, leg, comuns, k=10)
glossa = {p[0]: p for p in LEGAL.polysemous}
print(f"Jaccard@10 médio — polissêmicas jurídicas: {shift['jaccard@k'].mean():.3f} | palavras comuns: {ctrl['jaccard@k'].mean():.3f}")
fig = px.bar(pd.concat([shift.assign(grupo="polissêmicas jurídicas"), ctrl.assign(grupo="comuns")]), x="palavra", y="jaccard@k", color="grupo", template="plotly_white",
             title="Sobreposição das vizinhanças Wikipédia × STF por palavra (menor = maior deslocamento de domínio)")
fig.update_layout(xaxis_tickangle=-40); fig.show()

Jaccard@10 médio — polissêmicas jurídicas: 0.008 | palavras comuns: 0.041


As palavras com Jaccard ≈ 0 são as que **mudam de sentido** ao entrar no Direito: *sentença* (frase → decisão), *título* (nome → documento de crédito), *pena* (dó → sanção), *trânsito* (tráfego → trânsito em julgado), *remédio* (medicamento → remédio constitucional). O glossário:

In [9]:
pd.DataFrame([{"palavra": w, "sentido geral": g, "sentido jurídico": j} for w, g, j in LEGAL.polysemous if w in shift.palavra.values]).head(20)

,palavra,sentido geral,sentido jurídico
0,sentença,"frase, oração",decisão do juiz que põe fim à fase de conhecimento
1,ação,ato de agir; cota de empresa,"demanda judicial (ação penal, ação civil pública)"
2,parte,"porção, pedaço","sujeito do processo (autor, réu)"
3,título,nome de obra; cabeçalho,"documento que materializa um direito (título de crédito, título executivo)"
4,pena,dó; pluma,sanção penal
5,instrumento,ferramenta; instrumento musical,documento formal; agravo de instrumento
6,competência,habilidade,atribuição de um órgão para julgar
7,prescrição,receita médica; ordem,perda da pretensão pelo decurso do tempo
8,mérito,"valor, merecimento",questão de fundo da causa
9,tutela,"proteção, guarda","tutela jurisdicional, tutela de urgência"


## 4. Analogias: as gerais perdem cobertura; as jurídicas aparecem

O LX-4WAnalogies (capitais, moedas, família) é um teste "da Wikipédia". No modelo jurídico a **cobertura** cai para poucos por cento e a acurácia fica sem sentido — reforçando a conclusão de Hartmann et al. Testamos então relações próprias do domínio (*autor : réu :: apelante : apelado*, *juiz : sentença :: tribunal : acórdão*, *civil : CPC :: penal : CPP*). Expectativa honesta: com 6 M tokens e 100 dimensões, poucas analogias "fecham" no top-5 — mas as que fecham (*civil:cpc::penal:cpp*, *réu:ré::autor:autora*) só fecham no modelo **jurídico**, e os candidatos errados do STF são *do domínio* (*acusado, ré, acordão, procedente*), enquanto os da Wikipédia são aleatórios (*bemol, atari*).

In [10]:
from replang.data.analogies import load_analogies
from replang.eval.analogies import evaluate_analogies
an = load_analogies("pt-br")
rows = []
for wv in (pt, leg, mac):
    r = evaluate_analogies(wv, an, restrict=30000, max_per_category=300).set_index("categoria")
    rows.append({"modelo": wv.name, "cobertura": r.loc["TOTAL", "cobertas"] / r.loc["TOTAL", "n"], "acurácia (cobertas)": r.loc["TOTAL", "acurácia"]})
pd.DataFrame(rows).style.format({"cobertura": "{:.1%}", "acurácia (cobertas)": "{:.1%}"})

,modelo,cobertura,acurácia (cobertas)
0,Wikipédia,39.2%,56.3%
1,STF,6.6%,1.7%
2,Machado,21.5%,5.1%


In [11]:
rows = []
for a, b, c, d in LEGAL.analogies:
    for wv in (pt, leg):
        if wv.has(a, b, c):
            top = [x for x, _ in wv.analogy(a, b, c, topn=5)]
            rows.append({"analogia": f"{a} : {b} :: {c} : ?", "esperado": d, "modelo": wv.name, "top-5": ", ".join(top), "acerto@5": d in top})
        else:
            rows.append({"analogia": f"{a} : {b} :: {c} : ?", "esperado": d, "modelo": wv.name, "top-5": "OOV", "acerto@5": False})
df = pd.DataFrame(rows)
print(df.groupby("modelo")["acerto@5"].mean().rename("acerto@5 nas analogias jurídicas"))
df

modelo
STF          0.250000
Wikipédia    0.083333
Name: acerto@5 nas analogias jurídicas, dtype: float64


,analogia,esperado,modelo,top-5,acerto@5
0,autor : réu :: apelante : ?,apelado,Wikipédia,OOV,False
1,autor : réu :: apelante : ?,apelado,STF,"acusado, acusada, paciente, confessou, corré",False
2,juiz : sentença :: tribunal : ?,acórdão,Wikipédia,"condenação, instância, sentenças, nulidade, declaração",False
3,juiz : sentença :: tribunal : ?,acórdão,STF,"julgado, gaúcho, confirmatório, acórdão, reformou",True
4,juiz : sentença :: relator : ?,voto,Wikipédia,"tramitação, condenação, inconstitucionalidade, emendas, revogação",False
5,juiz : sentença :: relator : ?,voto,STF,"relatoria, edsegundos, agr-segundo-ed, 000.000:, agr-ed-edv-agr-ed",False
6,civil : cpc :: penal : ?,cpp,Wikipédia,"spectrum, stj, disciplinar, commodore, atari",False
7,civil : cpc :: penal : ?,cpp,STF,"cpp, auto-aplicabilidade, lvii, atenuação, figurino",True
8,réu : ré :: autor : ?,autora,Wikipédia,"bemol, sustenido, dó, debussy, fá",False
9,réu : ré :: autor : ?,autora,STF,"autora, apelante, licenciamento, árvore, apelada",True


## 5. Avaliação extrínseca I: NER jurídico (LeNER-Br)

Como no notebook 07 (POS), usamos janela de embeddings + regressão logística para prever as etiquetas BIO do LeNER-Br. O que importa é a **comparação** geral × jurídico × fastText (OOV). Métrica: F1 por token nas entidades (ignorando `O`), além da acurácia.

In [12]:
import time
from replang.data.legal import load_lener
from replang.eval.extrinsic import pos_tagging_eval
from replang.models.trainers import load_fasttext_model
from sklearn.metrics import f1_score
train = load_lener("train"); test = load_lener("test")
tags = pd.Series([t for s in train for _, t in s]).value_counts()
print(f"treino: {len(train)} sentenças / {tags.sum():,} tokens · teste: {len(test)} sentenças"); print(tags.head(8).to_dict())
ftm = load_fasttext_model("legal_ft100"); ft_leg = load_local_model("legal_ft100"); ft_leg.name = "STF fastText"
from replang.eval.parallel import parallel_map
from replang.eval.extrinsic import _window_features
def _ner(item):
    wv, use_ft = item
    oov_fn = (lambda w: ftm.wv[w]) if (use_ft and ftm is not None) else None
    t = time.time()
    r = pos_tagging_eval(wv, train, test, window=2, max_train_tokens=60000 if not FAST else 25000, oov_fn=oov_fn)
    Xte = np.vstack([_window_features(wv, [w for w, _ in s], 2, oov_fn) for s in test]); yte = np.array([t_ for s in test for _, t_ in s])
    pred = r["clf"].predict(Xte)
    ents = [t_ for t_ in sorted(set(yte)) if t_ != "O"]
    return {"embedding": wv.name, "acurácia": r["acurácia"], "F1 entidades (macro, token)": f1_score(yte, pred, labels=ents, average="macro"), "taxa OOV": r["taxa_oov"], "s": round(time.time() - t)}
rows = parallel_map(_ner, [(pt, False), (mac, False), (leg, False), (ft_leg, True)])
ner = pd.DataFrame(rows)
ner.style.format({"acurácia": "{:.3f}", "F1 entidades (macro, token)": "{:.3f}", "taxa OOV": "{:.1%}"}).background_gradient(subset=["F1 entidades (macro, token)"], cmap="Greens")

treino: 7827 sentenças / 229,277 tokens · teste: 1389 sentenças
{'O': 197228, 'I-LEGISLACAO': 11119, 'I-ORGANIZACAO': 4271, 'I-PESSOA': 3087, 'I-JURISPRUDENCIA': 2863, 'B-ORGANIZACAO': 2400, 'B-LEGISLACAO': 1920, 'B-PESSOA': 1525}


,embedding,acurácia,"F1 entidades (macro, token)",taxa OOV,s
0,Wikipédia,0.941,0.624,24.9%,5
1,Machado,0.914,0.424,35.8%,5
2,STF,0.941,0.580,23.3%,5
3,STF fastText,0.943,0.595,23.3%,6


## 6. Avaliação extrínseca II: similaridade de ementas (JurisBERT STS)

O "ASSIN jurídico": pares de ementas do STJ e do TJMS rotulados como **similares** (mesmo grupo temático) ou não. Protocolo: vetor da ementa = soma dos vetores das palavras; cosseno; medimos a capacidade de separar similares de não similares (AUC) e comparamos com TF-IDF.

In [13]:
from replang.data.legal import load_legal_sts
from replang.eval.extrinsic import sentence_vector
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
sts = load_legal_sts()
print(sts.shape, sts.tribunal.value_counts().to_dict(), "similar:", sts.similar.mean())
rows = []
for wv, oov_fn in [(pt, None), (mac, None), (leg, None), (ft_leg, (lambda w: ftm.wv[w]) if ftm is not None else None)]:
    cos = np.array([WordVectors.cosine(sentence_vector(wv, a, mode="mean", oov_fn=oov_fn), sentence_vector(wv, b, mode="mean", oov_fn=oov_fn)) for a, b in zip(sts.ementa1, sts.ementa2)])
    rows.append({"representação": wv.name, "AUC (similar × não similar)": roc_auc_score(sts.similar, cos), "cos médio similares": cos[sts.similar == 1].mean(), "cos médio não similares": cos[sts.similar == 0].mean()})
vec = TfidfVectorizer(min_df=2).fit(list(sts.ementa1) + list(sts.ementa2))
A, B = vec.transform(sts.ementa1), vec.transform(sts.ementa2)
cos_tfidf = np.asarray(A.multiply(B).sum(1)).ravel() / (np.sqrt(np.asarray(A.multiply(A).sum(1)).ravel()) * np.sqrt(np.asarray(B.multiply(B).sum(1)).ravel()) + 1e-9)
rows.append({"representação": "TF-IDF", "AUC (similar × não similar)": roc_auc_score(sts.similar, cos_tfidf), "cos médio similares": cos_tfidf[sts.similar == 1].mean(), "cos médio não similares": cos_tfidf[sts.similar == 0].mean()})
pd.DataFrame(rows).style.format({c: "{:.3f}" for c in rows[0] if c != "representação"}).background_gradient(subset=["AUC (similar × não similar)"], cmap="Greens")

(1200, 5) {'TJMS': 600, 'STJ': 600} similar: 0.5


,representação,AUC (similar × não similar),cos médio similares,cos médio não similares
0,Wikipédia,0.750,0.978,0.966
1,Machado,0.669,0.986,0.981
2,STF,0.810,0.963,0.931
3,STF fastText,0.822,0.964,0.931
4,TF-IDF,0.911,0.283,0.094


## 7. Avaliação extrínseca III: classificar a área do Direito pela ementa (RulingBR)

A tarefa do projeto Victor (STF) em miniatura: dada a ementa, prever a **área** (penal, administrativo, tributário…). Média de vetores → regressão logística, validação cruzada; comparamos geral × jurídico × TF-IDF.

In [14]:
from replang.data.legal import load_rulingbr_ementas
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
em = load_rulingbr_ementas()
top_areas = em.area.value_counts().head(6).index
em = em[em.area.isin(top_areas)].reset_index(drop=True)
y = em.area.to_numpy(dtype=str)
print(f"{len(em)} ementas em {len(top_areas)} áreas:", em.area.value_counts().to_dict())
rows = []
for wv in (pt, mac, leg):
    X = np.stack([sentence_vector(wv, e, mode="mean") for e in em.ementa])
    acc = cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=5).mean()
    rows.append({"representação": wv.name, "acurácia (CV 5)": acc})
Xt = TfidfVectorizer(min_df=3, max_features=30000).fit_transform(em.ementa)
rows.append({"representação": "TF-IDF", "acurácia (CV 5)": cross_val_score(LogisticRegression(max_iter=2000), Xt, y, cv=5).mean()})
pd.DataFrame(rows).style.format({"acurácia (CV 5)": "{:.1%}"}).background_gradient(subset=["acurácia (CV 5)"], cmap="Greens")

3000 ementas em 6 áreas: {'direito administrativo': 906, 'direito processual civil': 652, 'direito penal': 558, 'direito constitucional': 411, 'direito tributário': 299, 'direito civil': 174}


,representação,acurácia (CV 5)
0,Wikipédia,63.9%
1,Machado,45.7%
2,STF,67.2%
3,TF-IDF,78.5%


## 8. Subpalavras no jurídico: papéis em *-ante/-ado* e termos longos

A morfologia jurídica é regular: *agravante/agravado*, *embargante/embargado*, *impetrante/impetrado*, *apelante/apelado*, *recorrente/recorrido*. O fastText aprende a estrutura e cobre OOV (siglas, compostos).

In [15]:
pares = [("agravante", "agravado"), ("embargante", "embargado"), ("impetrante", "impetrado"), ("apelante", "apelado"), ("recorrente", "recorrido"), ("exequente", "executado"), ("locador", "locatário")]
rows = [{"par": f"{a}–{b}", "STF SG": leg.similarity(a, b) if leg.has(a, b) else np.nan, "STF fastText": float(ftm.wv.similarity(a, b)) if ftm else np.nan, "Wikipédia": pt.similarity(a, b) if pt.has(a, b) else np.nan} for a, b in pares]
display(pd.DataFrame(rows).style.format({c: "{:.2f}" for c in ["STF SG", "STF fastText", "Wikipédia"]}))
for w in ["inconstitucionalidade", "desconsideração", "repercussão", "hipossuficiência", "adpf"]:
    print(f"{w:<22} fastText STF: {[x for x, _ in ftm.wv.most_similar(w, topn=6)]}")

,par,STF SG,STF fastText,Wikipédia
0,agravante–agravado,0.71,0.78,0.63
1,embargante–embargado,0.70,0.79,nan
2,impetrante–impetrado,0.43,0.58,nan
3,apelante–apelado,0.71,0.79,nan
4,recorrente–recorrido,0.54,0.62,0.26
5,exequente–executado,0.70,0.60,nan
6,locador–locatário,nan,0.64,nan


inconstitucionalidade  fastText STF: ['inconstitucionalidades', 'inconstitucionalização', 'inconstitucional', 'constitucionalidade', 'inconstitucionalmente', 'inconstitucionais']
desconsideração        fastText STF: ['desconsidera', 'consideração', 'desconsiderando', 'desconsiderar', 'desconsiderado', 'reconsideração']
repercussão            fastText STF: ['geral', 'repercussões', 'mg-rg', 're-rg', 'rj-rg', 'rn-rg']
hipossuficiência       fastText STF: ['suficiência', 'ineficiência', 'insuficiência', 'hipossuficiente', 'deficiência', 'eficiência']
adpf                   fastText STF: ['adi', 'adc', 'hegemonia', 'adi-mc-ref', 'mc-ed-r', 'estudei']


## 9. Frases e composição aditiva no domínio

`word2phrase` sobre o corpus do STF encontra os sintagmas que o jurista usa como unidades: *habeas_corpus*, *repercussão_geral*, *dano_moral*, *devido_processo*. E a soma de vetores: *dano + moral*, *recurso + extraordinário*.

In [16]:
from replang.models.word2vec_np import learn_phrases
ph, scores = learn_phrases(jur, min_count=20, threshold=60, passes=1)
top = sorted(scores.items(), key=lambda x: -x[1])
print([p for p, _ in top[:40]])
def soma(wv, *ws, topn=6):
    v = sum(wv.unit[wv.index[w]] for w in ws); sims = wv.similarities(v)
    for w in ws: sims[wv.index[w]] = -np.inf
    return [wv.words[i] for i in np.argsort(-sims)[:topn]]
for ws in [("dano", "moral"), ("recurso", "extraordinário"), ("prisão", "preventiva"), ("união", "estável")]:
    print(f"{' + '.join(ws):<24} STF: {soma(leg, *ws)}   | Wikipédia: {soma(pt, *ws) if pt.has(*ws) else 'OOV'}")

['on_line', 'sydney_sanches', 'fumus_boni', 'ab_initio', 'modus_operandi', 'mc-r_ef', 'regimentos_internos', 'amicus_curiae', 'mato_grosso', 'interpreta_razoavelmente', 'erga_omnes', 'ag_reg', 'rodrigo_janot', 'santa_catarina', 'djaci_falcão', 'falsidade_ideológica', 'atentado_violento', 'candidaturas_avulsas', 'maurício_corrêa', 'ilmar_galvão', 'andré_lazaroni', 'alienados_fiduciariamente', 'confissão_espontânea', 'energia_elétrica', 'interceptação_telefônica', 'xxiii_xxiv', 'economia_mista', 'sub_examine', 'candidatura_avulsa', 'espírito_santo', 'janot_monteiro', 'unidades_autônomas', 'honorários_advocatícios', 'fático_delineado', 'sub_judice', 'verba_honorária', 'ement_vol', 'corretamente_assentadas', 'assis_moura', 'moreira_alves']
dano + moral             STF: ['causado', 'extrapatrimonial', 'causador', 'indenizável', 'abalo', 'sofrido']   | Wikipédia: ['sofrimento', 'constrangimento', 'emocional', 'temor', 'psicológica', 'intrínseco']
recurso + extraordinário STF: ['agravo', 'rec

## 10. Síntese: o que o domínio muda

1. **Vocabulário e forma**: sentenças mais longas, numerais e siglas como entidades, latim; o pré-processamento genérico destrói informação jurídica.
2. **Semântica**: palavras polissêmicas deslocam-se por completo (Jaccard ≈ 0); embeddings gerais "não sabem Direito" nem com 200 M tokens de Wikipédia.
3. **Avaliação**: a lição de Hartmann et al. vale em dobro — analogias gerais são irrelevantes (cobertura de 7 % no STF); nas tarefas jurídicas o embedding **de domínio** vence o geral (STS de ementas: AUC 0,81–0,82 × 0,75; área: 67 % × 64 %; NER: empate técnico, ambos muito acima do literário) e o **fastText** ajuda com OOV; mas o **TF-IDF é o melhor** em STS (0,91) e em área (78 %), porque o vocabulário técnico já é discriminativo e as ementas compartilham termos exatos. Embeddings de palavras somados **não** são o estado da arte para similaridade de documentos jurídicos — modelos contextuais (JurisBERT) são.
4. **Aplicações** (ver `docs/juridico.md` §5): pesquisa de jurisprudência, triagem/classificação (Victor), agrupamento de recursos repetitivos (Athos), extração de entidades, sumarização/linguagem simples.

### Perguntas para discussão
* A representação ideal para busca de precedentes é a que maximiza AUC em STS — ou a que o jurista consegue *explicar*?
* Um embedding treinado só em decisões do STF serve para um Juizado Especial? (deslocamento *dentro* do domínio: tribunal, instância, ano)
* O que fazer com os números: tokens, entidades ou ruído?

## Referências
- HARTMANN, N. et al. **Portuguese Word Embeddings: Evaluating on Word Analogies and Natural Language Tasks.** STIL, 2017. [arXiv:1708.06025](https://arxiv.org/abs/1708.06025)
- FARUQUI, M. et al. **Problems With Evaluation of Word Embeddings Using Word Similarity Tasks.** RepEval, 2016.
- BOJANOWSKI, P.; GRAVE, E.; JOULIN, A.; MIKOLOV, T. **Enriching Word Vectors with Subword Information.** TACL, 2017. [arXiv:1607.04606](https://arxiv.org/abs/1607.04606)
- MIKOLOV, T.; SUTSKEVER, I.; CHEN, K.; CORRADO, G.; DEAN, J. **Distributed Representations of Words and Phrases and their Compositionality.** NIPS, 2013. [arXiv:1310.4546](https://arxiv.org/abs/1310.4546)
- FEIJÓ, D.; MOREIRA, V. **RulingBR: A Summarization Dataset for Legal Texts.** PROPOR, 2018.
- LUZ DE ARAUJO, P. H. et al. **LeNER-Br: a Dataset for Named Entity Recognition in Brazilian Legal Text.** PROPOR, 2018.
- VIEGAS, C. F. O.; COSTA, B. C.; ISHII, R. P. **JurisBERT: A New Approach that Converts a Classification Corpus into an STS One.** ICCSA, 2023.
- POLO, F. M. et al. **LegalNLP.** ENIAC, 2021.